# UMUD Challenge: Complete Structured Ultrasound Pipeline
### Architecture: Modules A through G

This notebook implements the validated modular ultrasound muscle architecture pipeline:
- **Module A: Scan-Area Detector** — Local variance texture segmentation to isolate the active sector and ignore headers, UI text, and black borders.
- **Module B: Scale Extractor** — Measures ruler tick spacing and depth mapping to obtain calibrated mm/pixel per image.
- **Module C: Aponeurosis Detector** — Horizontal ridge enhancement, candidate pair scoring, and robust Huber line fitting.
- **Module D: Fascicle Analyzer** — Structure tensor gradient orientation field inside the muscle belly.
- **Module E: Fusion & Reliability Checks** — Enforces anatomical bounds and geometric consistency.
- **Module F: Trigonometry** — MT = Perpendicular distance, PA = angle between fiber and deep aponeurosis, FL = MT / sin(PA).
- **Module G: Submission Generation** — Generates complete 309-row `submission.csv` with semicolon (`;`) delimiter.


In [ ]:
import os, glob, math, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Set random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Detect search directories (works both locally and on Kaggle)
SEARCH_ROOTS = [
    '/kaggle/input',
    '/kaggle/input/competitions',
    '/kaggle/input/umud-challenge-muscle-architecture-in-ultrasound-data',
    '.',
    '..'
]

def find_dir(patterns):
    if isinstance(patterns, str):
        patterns = [patterns]
    for r in SEARCH_ROOTS:
        if not os.path.exists(r):
            continue
        for cur, dirs, _ in os.walk(r):
            for d in dirs:
                dl = d.lower()
                if any(p.lower() in dl for p in patterns):
                    target = os.path.join(cur, d)
                    files = [f for f in os.listdir(target) if f.lower().endswith(('.tif', '.png', '.jpg'))]
                    if len(files) > 0:
                        return target
    return None

APO_IMG_DIR  = find_dir(['apo_images_new_model_v1', 'apo_imgs_v1'])
APO_MSK_DIR  = find_dir(['apo_masks_new_model_v1', 'apo_masks_v1'])
TEST_IMG_DIR = find_dir(['test_set_v2', 'test_images_v2', 'test'])
OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
os.makedirs(OUT_DIR, exist_ok=True)

print('APO Images Dir :', APO_IMG_DIR)
print('APO Masks Dir  :', APO_MSK_DIR)
print('TEST Images Dir:', TEST_IMG_DIR)
print('Output Dir     :', OUT_DIR)

def list_valid_images(folder):
    if not folder or not os.path.exists(folder):
        return []
    exts = ('.tif', '.tiff', '.png', '.jpg', '.jpeg', '.bmp')
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(exts) and not f.startswith('.')])

test_files = list_valid_images(TEST_IMG_DIR)
print(f'Discovered {len(test_files)} test images.')


In [ ]:
# =====================================================================
# MODULE A : Scan-Area Detector
# =====================================================================
def detect_scan_area(img, ksize=21, thresh_std=8.0):
    """
    Detects the ultrasound rectangle and ignores UI text, color bars, and black borders.
    Uses local standard deviation (speckle texture is high, flat panels and borders are low).
    Returns (x0, y0, x1, y1), scan_width, scan_height.
    """
    h, w = img.shape
    mean = cv2.blur(img.astype(np.float32), (ksize, ksize))
    sq_mean = cv2.blur((img.astype(np.float32))**2, (ksize, ksize))
    local_std = np.sqrt(np.maximum(sq_mean - mean**2, 0))

    # Binary mask of textured ultrasound speckle
    mask = (local_std > thresh_std).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Largest connected component represents the active scan sector
    n, lbl, stats, cent = cv2.connectedComponentsWithStats(closed)
    if n > 1:
        largest_idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        x0, y0, bw, bh = stats[largest_idx, :4]
        x1, y1 = x0 + bw, y0 + bh
        # Sanity check: must cover at least 30% of the image
        if bw * bh < 0.3 * w * h:
            x0, y0, x1, y1 = 0, 0, w, h
            bw, bh = w, h
    else:
        x0, y0, x1, y1 = 0, 0, w, h
        bw, bh = w, h

    return (x0, y0, x1, y1), bw, bh

# Test on first available image
sample_img_path = os.path.join(TEST_IMG_DIR, test_files[0]) if test_files else None
if sample_img_path:
    test_img = cv2.imread(sample_img_path, cv2.IMREAD_GRAYSCALE)
    box, bw, bh = detect_scan_area(test_img)
    print(f'Sample {test_files[0]}: Image {test_img.shape[1]}x{test_img.shape[0]} -> Scan Box: {box}, Height: {bh}px')


In [ ]:
# =====================================================================
# MODULE B : Scale Extractor (mm/pixel & Ruler Tick Cross-Check)
# =====================================================================
# Known standard ultrasound depths and calibrated cluster profiles
CLUSTER_DEPTHS = {
    (1200, 800): 50.0,
    (1640, 1080): 45.0,
    (1088, 644): 45.0,
    (1069, 853): 50.0,
    (959, 853):  50.0,
    (760, 500):  50.0,
    (512, 512):  40.0,
    (465, 513):  40.0,
    (461, 513):  40.0,
}

def extract_ruler_pitch_px(img, box):
    """Scans image border margins for periodic ruler ticks."""
    x0, y0, x1, y1 = box
    h, w = img.shape
    best_spacing = None
    best_peaks = []
    
    # Check left margin (0..50) and right margin (W-60..W)
    search_cols = list(range(0, min(50, w))) + list(range(max(0, w-60), w))
    for c in search_cols:
        col_data = img[y0:y1, c].astype(float)
        if col_data.max() - col_data.min() < 30:
            continue
        peaks, _ = find_peaks(col_data, height=40, distance=10, prominence=15)
        if len(peaks) >= 5:
            diffs = np.diff(peaks)
            if np.std(diffs) < 3.5:
                if best_spacing is None or len(peaks) > len(best_peaks):
                    best_spacing = float(np.median(diffs))
                    best_peaks = peaks
    return best_spacing

def get_image_scale(img, box, w, h):
    """
    Computes mm_per_px using scan height and tick cross-check.
    """
    bh = box[3] - box[1]
    depth_mm = CLUSTER_DEPTHS.get((w, h), 50.0)
    
    # Scale from scan region height
    mm_per_px_depth = depth_mm / max(bh, 1)
    
    # Tick cross-check
    tick_pitch = extract_ruler_pitch_px(img, box)
    if tick_pitch is not None:
        if 10 <= tick_pitch <= 25:
            mmpx_tick = 1.0 / tick_pitch
        elif 60 <= tick_pitch <= 85:
            mmpx_tick = 5.0 / tick_pitch
        else:
            mmpx_tick = 10.0 / tick_pitch
            
        # If tick spacing agrees within 8% of depth scale, blend them
        if abs(mmpx_tick - mm_per_px_depth) / mm_per_px_depth < 0.08:
            return 0.5 * mm_per_px_depth + 0.5 * mmpx_tick, 'tick_crosscheck'
            
    return mm_per_px_depth, 'scan_height_depth'

print('Module B: Scale extractor configured.')


In [ ]:
# =====================================================================
# MODULE C : Aponeurosis Detector (OpenCV Ridge & Line Fitting)
# =====================================================================
_clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))

def detect_aponeuroses(cropped_img):
    """
    Detects superficial (top) and deep (bottom) aponeuroses.
    Enhances bright horizontal ridges, extracts candidate bands,
    scores candidate pairs, and fits robust straight lines (Huber).
    """
    ch, cw = cropped_img.shape
    # 1. Contrast enhancement
    eq = _clahe.apply(cropped_img)
    bl = cv2.GaussianBlur(eq, (5, 5), 1.0)
    
    # 2. Horizontal ridge enhancement (Sobel vertical derivative & closing)
    grad_y = cv2.Sobel(bl, cv2.CV_32F, 0, 1, ksize=3)
    ridge = np.abs(grad_y)
    ridge = cv2.normalize(ridge, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # 3. Horizontal morphological structuring
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (int(cw * 0.08), 3))
    morph = cv2.morphologyEx(ridge, cv2.MORPH_CLOSE, h_kernel)
    thresh = (morph > np.percentile(morph, 85)).astype(np.uint8)
    
    # 4. Connected components
    n, lbl, stats, cent = cv2.connectedComponentsWithStats(thresh)
    candidates = []
    for i in range(1, n):
        w_comp = stats[i, cv2.CC_STAT_WIDTH]
        if w_comp >= 0.25 * cw:
            candidates.append((i, stats[i, cv2.CC_STAT_AREA], cent[i][1]))
            
    if len(candidates) < 2:
        # Fallback to standard anatomical top and bottom boundaries
        sup_line = (0.0, float(ch * 0.20))
        dep_line = (0.0, float(ch * 0.80))
        return sup_line, dep_line, 0, cw
        
    # Sort candidates top-to-bottom
    candidates = sorted(candidates, key=lambda c: c[2])
    top_cands = [c for c in candidates if c[2] < ch * 0.50]
    bot_cands = [c for c in candidates if c[2] >= ch * 0.40]
    
    best_top = max(top_cands, key=lambda c: c[1])[0] if top_cands else candidates[0][0]
    best_bot = max(bot_cands, key=lambda c: c[1])[0] if bot_cands else candidates[-1][0]
    
    # Fit robust lines
    lines = []
    x_ranges = []
    for idx in (best_top, best_bot):
        ys, xs = np.nonzero(lbl == idx)
        pts = np.column_stack([xs, ys]).astype(np.float32).reshape(-1, 1, 2)
        vx, vy, x0, y0 = cv2.fitLine(pts, cv2.DIST_HUBER, 0, 0.01, 0.01).flatten()
        m = float(vy / vx) if abs(vx) > 1e-6 else 0.0
        m = float(np.clip(m, -0.45, 0.45))
        b = float(y0 - m * x0)
        lines.append((m, b))
        x_ranges.append((xs.min(), xs.max()))
        
    x_lo = max(x_ranges[0][0], x_ranges[1][0])
    x_hi = min(x_ranges[0][1], x_ranges[1][1])
    if x_hi - x_lo < 0.2 * cw:
        x_lo, x_hi = 0, cw
        
    return lines[0], lines[1], x_lo, x_hi

print('Module C: Aponeurosis detector configured.')


In [ ]:
# =====================================================================
# MODULE D : Fascicle Orientation & Fiber Direction
# =====================================================================
def detect_fascicle_angle(cropped_img, sup_line, dep_line, x_lo, x_hi):
    """
    Computes muscle fiber direction via Structure Tensor inside the inter-aponeurotic belly.
    Returns pennation angle (degrees relative to deep aponeurosis).
    """
    ch, cw = cropped_img.shape
    bl = cv2.GaussianBlur(cropped_img, (3, 3), 1.0)
    gx = cv2.Sobel(bl, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(bl, cv2.CV_32F, 0, 1, ksize=3)
    
    # Structure tensor components
    Jxx = cv2.GaussianBlur(gx * gx, (7, 7), 2.5)
    Jyy = cv2.GaussianBlur(gy * gy, (7, 7), 2.5)
    Jxy = cv2.GaussianBlur(gx * gy, (7, 7), 2.5)
    coherence = np.sqrt((Jxx - Jyy)**2 + 4 * Jxy**2) / (Jxx + Jyy + 1e-6)
    
    # Create mask of muscle belly between the two lines
    xs = np.arange(cw, dtype=np.float32)
    ys = np.arange(ch, dtype=np.float32)
    XS, YS = np.meshgrid(xs, ys)
    
    y_sup = sup_line[0] * XS + sup_line[1]
    y_dep = dep_line[0] * XS + dep_line[1]
    
    # 12% safety margin away from aponeurosis lines
    thickness = np.maximum(y_dep - y_sup, 1.0)
    margin = thickness * 0.12
    belly_mask = (YS >= y_sup + margin) & (YS <= y_dep - margin) & (XS >= x_lo) & (XS <= x_hi)
    
    weights = np.where(belly_mask, coherence, 0.0)
    w_sum = weights.sum()
    if w_sum < 1e-4:
        return 14.5
        
    avg_Jxx = (weights * Jxx).sum() / w_sum
    avg_Jyy = (weights * Jyy).sum() / w_sum
    avg_Jxy = (weights * Jxy).sum() / w_sum
    
    # Principal gradient direction
    theta_grad = 0.5 * math.atan2(2 * avg_Jxy, avg_Jxx - avg_Jyy)
    theta_fiber = theta_grad + math.pi / 2
    theta_fiber_deg = math.degrees(theta_fiber)
    
    theta_deep_deg = math.degrees(math.atan(dep_line[0]))
    pa = abs((theta_fiber_deg - theta_deep_deg + 90) % 180 - 90)
    return float(np.clip(pa, 3.0, 45.0))

print('Module D: Fascicle analyzer configured.')


In [ ]:
# =====================================================================
# MODULES E & F : Fusion, Geometric Validation & Trigonometry
# =====================================================================
def analyze_single_image(img_path):
    """
    Runs end-to-end pipeline:
    Image -> Crop Box -> mmpx -> Lines -> Fascicle Angle -> MT, PA, FL
    """
    raw = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if raw is None:
        return None
    h_orig, w_orig = raw.shape
    
    # Module A: Scan Area Detection
    box, bw, bh = detect_scan_area(raw)
    x0, y0, x1, y1 = box
    cropped = raw[y0:y1, x0:x1]
    
    # Module B: Scale Extraction
    mmpx, scale_src = get_image_scale(raw, box, w_orig, h_orig)
    
    # Module C: Aponeurosis Detection on cropped image
    sup_line, dep_line, x_lo, x_hi = detect_aponeuroses(cropped)
    
    # Perpendicular distance (Muscle Thickness in pixels)
    xs = np.linspace(x_lo, x_hi, 100)
    vert_dist = (dep_line[0] * xs + dep_line[1]) - (sup_line[0] * xs + sup_line[1])
    mean_vert = float(np.maximum(vert_dist.mean(), 10.0))
    mt_px = mean_vert / math.sqrt(1 + dep_line[0]**2)
    mt_mm = float(np.clip(mt_px * mmpx, 5.0, 60.0))
    
    # Module D: Fascicle Angle
    pa_deg = detect_fascicle_angle(cropped, sup_line, dep_line, x_lo, x_hi)
    
    # Module F: Trigonometry for Fascicle Length (FL = MT / sin(PA))
    fl_computed = mt_mm / math.sin(math.radians(pa_deg))
    fl_mm = float(np.clip(fl_computed, 20.0, 200.0))
    
    return {
        'pa_deg': round(pa_deg, 3),
        'fl_mm': round(fl_mm, 3),
        'mt_mm': round(mt_mm, 3),
        'box': box,
        'mmpx': mmpx,
        'scale_src': scale_src,
        'sup_line': sup_line,
        'dep_line': dep_line,
        'cropped': cropped
    }

print('Modules E & F: Trigonometric pipeline engine ready.')


In [ ]:
# Visual sanity check on training images if available
if APO_IMG_DIR:
    train_samples = list_valid_images(APO_IMG_DIR)[:3]
    fig, axes = plt.subplots(1, len(train_samples), figsize=(16, 5))
    if len(train_samples) == 1: axes = [axes]
    
    for ax, fname in zip(axes, train_samples):
        res = analyze_single_image(os.path.join(APO_IMG_DIR, fname))
        if res:
            vis = cv2.cvtColor(res['cropped'], cv2.COLOR_GRAY2BGR)
            cw = vis.shape[1]
            y_s0, y_s1 = int(res['sup_line'][1]), int(res['sup_line'][0] * cw + res['sup_line'][1])
            y_d0, y_d1 = int(res['dep_line'][1]), int(res['dep_line'][0] * cw + res['dep_line'][1])
            cv2.line(vis, (0, y_s0), (cw, y_s1), (0, 255, 255), 2)
            cv2.line(vis, (0, y_d0), (cw, y_d1), (255, 0, 255), 2)
            
            ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{fname}\nMT: {res['mt_mm']}mm | PA: {res['pa_deg']}° | FL: {res['fl_mm']}mm", fontsize=9)
            ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# =====================================================================
# MODULE G : Full Test Inference & Submission Generation
# =====================================================================
print(f'Starting inference on {len(test_files)} test images...')
results = []
failures = 0

for i, fname in enumerate(test_files):
    path = os.path.join(TEST_IMG_DIR, fname)
    res = analyze_single_image(path)
    if res is not None:
        results.append({
            'image_id': fname,
            'pa_deg': res['pa_deg'],
            'fl_mm': res['fl_mm'],
            'mt_mm': res['mt_mm']
        })
    else:
        failures += 1
        results.append({
            'image_id': fname,
            'pa_deg': 14.5,
            'fl_mm': 90.0,
            'mt_mm': 22.5
        })
    if (i + 1) % 50 == 0:
        print(f'  Processed {i + 1}/{len(test_files)} images')

sub_df = pd.DataFrame(results)
print(f'\nProcessed {len(sub_df)} total records ({failures} fallbacks).')

# Sanity assertion checks
assert len(sub_df) == len(test_files), f'Count mismatch: {len(sub_df)} vs {len(test_files)}'
assert sub_df['image_id'].is_unique, 'Duplicate image IDs found!'
assert not sub_df.isna().any().any(), 'NaN values detected!'

# Enforce consistent natural ordering
sub_df = sub_df.sort_values('image_id').reset_index(drop=True)

# Save submission with semicolon delimiter (as required by UMUD format)
SUBMISSION_PATH = os.path.join(OUT_DIR, 'submission.csv')
sub_df.to_csv(SUBMISSION_PATH, index=False, sep=';')

print(f'Successfully saved: {SUBMISSION_PATH}')
print(f'File size: {os.path.getsize(SUBMISSION_PATH)} bytes')
print('\nFirst 10 submission rows:')
print(sub_df.head(10).to_string(index=False))
print('\nSummary Statistics:')
print(sub_df.describe().to_string())
